# 🚀 Part 4 (Bonus) — Production Frameworks: Ollama · LangGraph · Deep Agents · a Packaged Harness
### Workshop 4 · LLMA4SE Summer School 2026

---

**⏱ Time:** ~60 min (self-paced / take-home) &nbsp;·&nbsp; **Needs:** T4 GPU runtime

In Parts 1–3 you built every pattern **by hand** — on purpose, so nothing is magic. This bonus part rebuilds the *same* pipeline with the tools you'd reach for on Monday morning:

| You hand-built (Parts 1–3) | Production equivalent (this notebook) |
|---|---|
| `llm()` helper over in-process `transformers` | **Ollama** — a local model *server* |
| `WorkflowState` blackboard dataclass | **LangGraph** typed `State` |
| `run_workflow()` retry loop | LangGraph **conditional edges** |
| `state.history` audit trail | LangGraph checkpointing / streaming |
| Auditor with tools + JSON contract | **Deep Agents** with real tool-calling, planning & sub-agents |
| a pile of notebook cells | a **lean, pip-installable CLI harness** (`debtbuster`) |

> **The punchline stays the same:** deterministic tools measure, LLMs interpret, gates decide. Frameworks change the *plumbing*, never the *principle*.

```
 §4.1 Ollama server ──► §4.2 LangGraph refactoring team ──► §4.3 Deep Agent
                                        │
                                        ▼
                    §4.4 debtbuster/ ── a packaged CLI harness you keep
```

## 4.0 · Setup — install Ollama on Colab (~5 min)

**Why a model server instead of `transformers` in-process?**
1. **Quantization for free** — Ollama ships 4-bit builds: a **7B** model in ~4.7 GB, so a *stronger brain* than Parts 1–3 fits the same T4.
2. **One model, many clients** — every agent, notebook and CLI talks to `localhost:11434`; the model loads **once**.
3. **A standard API** — swap `qwen2.5-coder` for any other model (or point the same code at a cloud endpoint) by changing **one string**.

In [ ]:
# 1) Install the Ollama binary and start the server in the background
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, requests, os
server = subprocess.Popen(["ollama", "serve"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):                       # wait until the server answers
    try:
        requests.get("http://localhost:11434", timeout=1); print("✅ Ollama server is up"); break
    except requests.exceptions.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start — re-run this cell")

In [ ]:
# 2) Pull the brain. 7B @ 4-bit ≈ 4.7 GB — a big upgrade over Part 1's 1.5B, still tiny on a T4.
#    Slow connection? Use "qwen2.5-coder:3b" everywhere instead (~1.9 GB).
OLLAMA_MODEL = "qwen2.5-coder:7b"
!ollama pull {OLLAMA_MODEL}
!ollama list

In [ ]:
# 3) Python-side packages + recreate the Part 1/2 patient (this notebook is standalone)
%pip install -q langgraph langchain-ollama deepagents radon pylint code-quality-analyzer pytest
%pip install -q git+https://github.com/KarthikShivasankar/ml_smells_detector.git
print("✅ packages ready")

In [ ]:
# Recreate the smelly module + its behaviour tests from Part 1
with open('inventory.py', 'w') as f:
    f.write('''"""inventory.py -- Order processing for a small e-commerce shop.

This module works correctly (all tests pass!) but it is deliberately
full of code smells. Your agents will find and fix them.
"""

def helper_unused(x):          # SMELL: dead code -- never called anywhere
    return x * 2


class InventoryManager:
    """Manages stock and processes customer orders."""

    def __init__(self, items=[]):              # SMELL: mutable default argument
        self.items = {}
        for name, price, qty in items:
            self.items[name] = {"price": price, "qty": qty}
        self.log = []

    def add_item(self, name, price, qty, category, supplier, discount, taxable):
        # SMELL: long parameter list (7 params, most unused)
        self.items[name] = {"price": price, "qty": qty}
        return True

    def process_order(self, order):
        # SMELL: long method, deep nesting, magic numbers, duplication
        total = 0.0
        status = "ok"
        for name, qty in order:
            if name in self.items:
                if self.items[name]["qty"] >= qty:
                    if qty > 0:
                        price = self.items[name]["price"]
                        subtotal = price * qty
                        if subtotal > 100:                      # magic number
                            subtotal = subtotal - subtotal * 0.05   # magic number
                        if qty > 10:                            # magic number
                            subtotal = subtotal - subtotal * 0.02   # magic number
                        total = total + subtotal
                        self.items[name]["qty"] = self.items[name]["qty"] - qty
                        self.log.append("sold " + name)
                    else:
                        status = "invalid_qty"
                else:
                    status = "insufficient_stock"
            else:
                status = "unknown_item"
        total = total + total * 0.25            # magic number (VAT)
        return {"total": round(total, 2), "status": status}

    def refund_order(self, order):
        # SMELL: duplicated logic (mirror of process_order maths)
        total = 0.0
        for name, qty in order:
            if name in self.items:
                price = self.items[name]["price"]
                subtotal = price * qty
                if subtotal > 100:                              # magic number again
                    subtotal = subtotal - subtotal * 0.05
                if qty > 10:
                    subtotal = subtotal - subtotal * 0.02
                total = total + subtotal
                self.items[name]["qty"] = self.items[name]["qty"] + qty
        total = total + total * 0.25
        return {"total": round(total, 2), "status": "refunded"}

    def get_stock(self, name):
        if name in self.items:
            return self.items[name]["qty"]
        return 0
''')
with open('test_inventory.py', 'w') as f:
    f.write('''"""test_inventory.py -- Behaviour-preserving safety net.

These tests define the PUBLIC CONTRACT of the module. Any refactoring
your agents perform MUST keep every one of these green.
"""
import pytest
from inventory import InventoryManager


@pytest.fixture
def mgr():
    return InventoryManager([("widget", 10.0, 100), ("gizmo", 25.0, 5)])


def test_simple_order(mgr):
    result = mgr.process_order([("widget", 2)])
    assert result["status"] == "ok"
    assert result["total"] == 25.0          # 20 + 25% VAT


def test_bulk_discount_applied(mgr):
    # 20 widgets = 200 -> -5% (>100) -> -2% (>10 units) -> +25% VAT
    result = mgr.process_order([("widget", 20)])
    assert result["total"] == 232.75


def test_stock_is_decremented(mgr):
    mgr.process_order([("widget", 2)])
    assert mgr.get_stock("widget") == 98


def test_insufficient_stock(mgr):
    result = mgr.process_order([("gizmo", 99)])
    assert result["status"] == "insufficient_stock"


def test_unknown_item(mgr):
    result = mgr.process_order([("nonexistent", 1)])
    assert result["status"] == "unknown_item"


def test_refund_restores_stock(mgr):
    mgr.process_order([("widget", 2)])
    mgr.refund_order([("widget", 2)])
    assert mgr.get_stock("widget") == 100


def test_no_shared_state_between_instances():
    a = InventoryManager()
    b = InventoryManager()
    a.items["x"] = {"price": 1, "qty": 1}
    assert "x" not in b.items or a.items is not b.items
''')
print('✅ target project recreated')
!python -m pytest test_inventory.py -q

## 4.1 · First contact — the same brain, now behind an API

`ChatOllama` is a LangChain chat model that talks to your local server. Note what we **don't** do anymore: no `AutoModelForCausalLM`, no chat template, no `.to("cuda")`. That's the server's job now.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.1)

print(llm.invoke("In one sentence: what is a code smell?").content)

> 🧪 **Try it (1 min):** run `!ollama ps` — see the model resident in GPU memory. Then invoke the LLM again: notice it's *fast* now (no reload). That's the server earning its keep.

## 4.2 · The refactoring team, rebuilt in **LangGraph**

LangGraph models a workflow as a **typed state + a graph of nodes**. Compare each line to what you wrote in Part 2 — it's a 1:1 translation:

- our `WorkflowState` dataclass → a `TypedDict` the graph carries between nodes
- our `run_workflow()` while-loop → a **conditional edge** (`qa → refactorer` on reject, `qa → END` on accept)
- our `max_iterations` guard → the same guard, expressed in the routing function

First, the deterministic pieces — copied straight from Part 2, because **the QA gate never changes, whatever the framework**:

In [ ]:
import subprocess, tempfile, pathlib, ast, re, shutil, json

def run_radon_cc(path: str) -> float:
    """Average cyclomatic complexity (gate 3 uses this)."""
    out = subprocess.run(["radon", "cc", "-s", "-j", path], capture_output=True, text=True).stdout
    data = json.loads(out or "{}")
    scores = [b["complexity"] for blocks in data.values() for b in blocks]
    return round(sum(scores) / len(scores), 2) if scores else 0.0

def extract_code_block(text: str) -> str:
    m = re.findall(r"```(?:python)?\s*(.*?)```", text, re.S)
    if not m:
        raise ValueError("no ```python``` block in reply")
    return m[-1].strip() + "\n"

def qa_verify(original_path: str, candidate: str) -> dict:
    """Gate 1 syntax → gate 2 behaviour tests in a sandbox → gate 3 complexity. NO LLM inside."""
    try:
        ast.parse(candidate)                                     # gate 1
    except SyntaxError as e:
        return {"ok": False, "why": f"gate 1 (syntax): {e}"}
    with tempfile.TemporaryDirectory() as tmp:                   # gate 2
        pathlib.Path(tmp, "inventory.py").write_text(candidate)
        shutil.copy("test_inventory.py", tmp)
        r = subprocess.run(["python", "-m", "pytest", "-q", "test_inventory.py"],
                           cwd=tmp, capture_output=True, text=True, timeout=120)
        if r.returncode != 0:
            return {"ok": False, "why": "gate 2 (behaviour): " + r.stdout[-300:]}
        cc_after = run_radon_cc(str(pathlib.Path(tmp, "inventory.py")))
    cc_before = run_radon_cc(original_path)                      # gate 3
    if cc_after > cc_before:
        return {"ok": False, "why": f"gate 3 (quality): CC worsened {cc_before} → {cc_after}"}
    return {"ok": True, "why": f"all gates passed · CC {cc_before} → {cc_after}"}

print("✅ deterministic gates ready · current avg CC:", run_radon_cc("inventory.py"))

Now the graph. Three nodes, one loop:

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

AUDITOR_SYS = ("You are a strict senior code reviewer. Given a Python module, list its "
               "worst code smells as short bullet points (max 6). Be specific: name the "
               "function and the problem. No fixes yet, no prose intro.")
REFACTORER_SYS = ("You are a careful refactoring engineer. Rewrite the module fixing the "
                  "listed smells. HARD INVARIANTS: same public API and behaviour; stdlib only; "
                  "replace magic numbers with named constants; remove duplication and dead code. "
                  "Reply with ONE ```python``` block containing the FULL module, nothing else.")

class TeamState(TypedDict):
    source: str          # current original module text
    findings: str        # auditor bullets
    candidate: str       # latest refactored module
    verdict: str         # QA explanation
    accepted: bool
    iteration: int

def auditor_node(state: TeamState) -> dict:
    print("🕵️ auditor …")
    reply = llm.invoke([("system", AUDITOR_SYS), ("user", state["source"])])
    return {"findings": reply.content}

def refactorer_node(state: TeamState) -> dict:
    print(f"🔧 refactorer (iteration {state['iteration'] + 1}) …")
    fb = f"\nPREVIOUS ATTEMPT REJECTED: {state['verdict']}" if state["verdict"] else ""
    reply = llm.invoke([("system", REFACTORER_SYS),
                        ("user", f"MODULE:\n```python\n{state['source']}\n```\n"
                                 f"SMELLS:\n{state['findings']}{fb}")])
    try:
        cand = extract_code_block(reply.content)
    except ValueError as e:
        return {"candidate": "", "verdict": str(e), "iteration": state["iteration"] + 1}
    return {"candidate": cand, "iteration": state["iteration"] + 1}

def qa_node(state: TeamState) -> dict:
    print("🛡️ qa gates …")
    if not state["candidate"]:
        return {"accepted": False}
    v = qa_verify("inventory.py", state["candidate"])
    print("   ", "✅" if v["ok"] else "❌", v["why"][:90])
    return {"accepted": v["ok"], "verdict": v["why"]}

def route_after_qa(state: TeamState) -> str:
    if state["accepted"]:
        return "done"
    return "give_up" if state["iteration"] >= 3 else "retry"

g = StateGraph(TeamState)
g.add_node("auditor", auditor_node)
g.add_node("refactorer", refactorer_node)
g.add_node("qa", qa_node)
g.add_edge(START, "auditor")
g.add_edge("auditor", "refactorer")
g.add_edge("refactorer", "qa")
g.add_conditional_edges("qa", route_after_qa, {"done": END, "retry": "refactorer", "give_up": END})
team = g.compile()
print("✅ graph compiled")

In [ ]:
# LangGraph draws its own architecture diagram — compare it to the slide from Part 2!
from IPython.display import Image, display
try:
    display(Image(team.get_graph().draw_mermaid_png()))
except Exception:
    print(team.get_graph().draw_mermaid())   # fallback: raw mermaid text

In [ ]:
result = team.invoke({"source": open("inventory.py").read(),
                      "findings": "", "candidate": "", "verdict": "",
                      "accepted": False, "iteration": 0})

print("\n" + "="*60)
if result["accepted"]:
    open("inventory_langgraph.py", "w").write(result["candidate"])
    print("🏆 accepted after", result["iteration"], "iteration(s) —", result["verdict"])
    print("saved → inventory_langgraph.py")
else:
    print("🛑 no candidate passed the gates —", result["verdict"][:200])
    print("(the gate held: nothing unverified ships. Re-run — temperature>0 varies attempts.)")

> 💬 **Compare (2 min):** open Part 2's `run_workflow()` next to this. Same auditor, same gates, same loop. What did LangGraph buy you? (Answers: the diagram for free, typed state, streaming/checkpointing when you need it, and a shape your colleagues already know.) What did it cost? (A dependency, and some magic.)

## 4.3 · **Deep Agents** — planning, file tools & sub-agents

Parts 1–3 agents followed *our* fixed loop. A **deep agent** plans its *own* loop: it gets a todo-list tool (`write_todos`), file tools (`ls`, `read_file`, `write_file`, …), can delegate to **sub-agents**, and decides the order of operations itself. This is real **tool-calling** — the model emits structured calls, not text we parse.

We hand it our three research detectors as custom tools and let it run an audit autonomously:

In [ ]:
from deepagents import create_deep_agent

def radon_report(path: str) -> str:
    """Cyclomatic-complexity report for a Python file or directory."""
    return subprocess.run(["radon", "cc", "-s", path], capture_output=True, text=True).stdout or "n/a"

def pyexamine_report(path: str) -> str:
    """PyExamine (MSR 2025): 49-metric code-smell report for a directory."""
    subprocess.run(["analyze_code_quality", path, "--type", "code", "--output", "pyx"],
                   capture_output=True, text=True, timeout=600)
    p = pathlib.Path("pyx.txt")
    return p.read_text()[:3000] if p.exists() else "no report produced"

def mlscent_report(path: str) -> str:
    """MLScent (CAIN 2025): 76 ML-specific anti-pattern detectors for a directory."""
    subprocess.run(["ml_smell_detector", "analyze", path], capture_output=True, text=True, timeout=300)
    p = pathlib.Path("output/analysis_report.txt")
    return p.read_text()[:3000] if p.exists() else "no report produced"

deep_auditor = create_deep_agent(
    model=llm,
    tools=[radon_report, pyexamine_report, mlscent_report],
    system_prompt=(
        "You are an autonomous code-quality auditor. Plan your work with write_todos first. "
        "Use radon_report and pyexamine_report on Python business code; use mlscent_report "
        "only if the code imports ML libraries. Read files before judging them. "
        "Finish by writing a concise AUDIT.md (max 25 lines) with your top findings, "
        "each with file, smell, and a one-line fix."
    ),
)
print("✅ deep agent ready — tools:", [t.__name__ for t in (radon_report, pyexamine_report, mlscent_report)])

In [ ]:
# Let it loose on the current directory. Watch the todo list appear, then tool calls.
run = deep_agent_result = deep_auditor.invoke(
    {"messages": [("user", "Audit the Python code in this directory (start with inventory.py) "
                           "and produce AUDIT.md.")]},
    config={"recursion_limit": 40},
)

for m in run["messages"]:
    kind = m.__class__.__name__
    if getattr(m, "tool_calls", None):
        for tc in m.tool_calls:
            print(f"🔩 tool call → {tc['name']}({str(tc['args'])[:60]})")
    elif kind == "ToolMessage":
        pass                                    # tool outputs are long — skip
print("\n--- final answer ---\n", run["messages"][-1].content[:800])

import pathlib
if pathlib.Path("AUDIT.md").exists():
    print("\n📄 AUDIT.md written:\n", pathlib.Path("AUDIT.md").read_text()[:800])

> ⚠️ **Reality check (say this out loud to yourself):** a 7B local model is at the *lower edge* of reliable autonomous tool-calling. Sometimes it plans beautifully; sometimes it skips `write_todos` or forgets to write `AUDIT.md`. **That variance is the lesson** — Parts 1–3 worked deterministically *because we owned the loop*. Deep agents trade control for autonomy; production systems mix both: deep agents for exploration, hard-coded graphs + gates for anything that ships. (Swap `model=llm` for a frontier model and watch reliability jump — the code doesn't change.)

## 4.4 · The lean coding harness: package it as **`debtbuster`** 📦

A workshop pipeline dies with the notebook. A **harness** survives: a minimal, pip-installable CLI your team can run in CI. Lean means: **5 source files, 1 config, tests, no framework beyond what we used.** We write it straight from the notebook:

In [ ]:
!mkdir -p debtbuster/src/debtbuster debtbuster/tests

In [ ]:
%%writefile debtbuster/pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "debtbuster"
version = "0.1.0"
description = "Lean LLM-agent harness: audit, refactor & gate Python code (LLMA4SE Workshop 4)"
requires-python = ">=3.10"
dependencies = [
  "langgraph", "langchain-ollama",
  "radon", "pylint", "code-quality-analyzer", "pytest",
]

[project.scripts]
debtbuster = "debtbuster.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

In [ ]:
%%writefile debtbuster/src/debtbuster/config.py
"""Single source of truth — change the brain here, nowhere else."""
OLLAMA_MODEL = "qwen2.5-coder:7b"
MAX_ITERATIONS = 3
TEMPERATURE = 0.1

In [ ]:
%%writefile debtbuster/src/debtbuster/tools.py
"""Deterministic eyes. No LLM in this file."""
import json, pathlib, subprocess

def radon_avg_cc(path: str) -> float:
    out = subprocess.run(["radon", "cc", "-s", "-j", path],
                         capture_output=True, text=True).stdout
    data = json.loads(out or "{}")
    scores = [b["complexity"] for blocks in data.values() for b in blocks]
    return round(sum(scores) / len(scores), 2) if scores else 0.0

def pyexamine(path: str) -> str:
    subprocess.run(["analyze_code_quality", path, "--type", "code", "--output", "pyx"],
                   capture_output=True, text=True, timeout=600)
    rpt = pathlib.Path("pyx.txt")
    return rpt.read_text()[:3000] if rpt.exists() else ""

In [ ]:
%%writefile debtbuster/src/debtbuster/gates.py
"""The QA gate. Deliberately boring, deliberately LLM-free."""
import ast, pathlib, shutil, subprocess, tempfile
from .tools import radon_avg_cc

def verify(original: str, candidate: str, test_file: str | None) -> dict:
    try:
        ast.parse(candidate)                                        # gate 1: syntax
    except SyntaxError as e:
        return {"ok": False, "why": f"gate 1 (syntax): {e}"}
    if test_file:                                                   # gate 2: behaviour
        with tempfile.TemporaryDirectory() as tmp:
            tgt = pathlib.Path(tmp, pathlib.Path(original).name)
            tgt.write_text(candidate)
            shutil.copy(test_file, tmp)
            r = subprocess.run(["python", "-m", "pytest", "-q", pathlib.Path(test_file).name],
                               cwd=tmp, capture_output=True, text=True, timeout=120)
            if r.returncode != 0:
                return {"ok": False, "why": "gate 2 (behaviour): " + r.stdout[-300:]}
    with tempfile.TemporaryDirectory() as tmp:                      # gate 3: quality
        p = pathlib.Path(tmp, "cand.py"); p.write_text(candidate)
        before, after = radon_avg_cc(original), radon_avg_cc(str(p))
    if after > before:
        return {"ok": False, "why": f"gate 3 (quality): CC {before} -> {after}"}
    return {"ok": True, "why": f"gates passed - CC {before} -> {after}"}

In [ ]:
%%writefile debtbuster/src/debtbuster/graph.py
"""The LangGraph team: auditor -> refactorer <-> gates. Mirrors the notebook 1:1."""
import re
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from . import config
from .gates import verify
from .tools import pyexamine

AUDITOR_SYS = ("You are a strict senior code reviewer. List the module's worst code "
               "smells as max 6 short bullets. Name function + problem. No fixes, no intro.")
REFACTORER_SYS = ("You are a careful refactoring engineer. Rewrite the module fixing the "
                  "listed smells. HARD INVARIANTS: same public API and behaviour; stdlib only; "
                  "named constants for magic numbers; remove duplication and dead code. "
                  "Reply with ONE ```python``` block with the FULL module, nothing else.")

class State(TypedDict):
    path: str; test_file: str | None; source: str
    findings: str; candidate: str; verdict: str
    accepted: bool; iteration: int

def _llm():
    return ChatOllama(model=config.OLLAMA_MODEL, temperature=config.TEMPERATURE)

def _extract(text: str) -> str:
    m = re.findall(r"```(?:python)?\s*(.*?)```", text, re.S)
    if not m:
        raise ValueError("no code block in reply")
    return m[-1].strip() + "\n"

def auditor(state: State) -> dict:
    evidence = pyexamine(str(__import__("pathlib").Path(state["path"]).parent))
    reply = _llm().invoke([("system", AUDITOR_SYS),
                           ("user", f"STATIC EVIDENCE:\n{evidence}\n\nMODULE:\n{state['source']}")])
    return {"findings": reply.content}

def refactorer(state: State) -> dict:
    fb = f"\nPREVIOUS ATTEMPT REJECTED: {state['verdict']}" if state["verdict"] else ""
    reply = _llm().invoke([("system", REFACTORER_SYS),
                           ("user", f"MODULE:\n```python\n{state['source']}\n```\n"
                                    f"SMELLS:\n{state['findings']}{fb}")])
    try:
        return {"candidate": _extract(reply.content), "iteration": state["iteration"] + 1}
    except ValueError as e:
        return {"candidate": "", "verdict": str(e), "iteration": state["iteration"] + 1}

def qa(state: State) -> dict:
    if not state["candidate"]:
        return {"accepted": False}
    v = verify(state["path"], state["candidate"], state["test_file"])
    return {"accepted": v["ok"], "verdict": v["why"]}

def _route(state: State) -> str:
    if state["accepted"]:
        return "done"
    return "give_up" if state["iteration"] >= config.MAX_ITERATIONS else "retry"

def build():
    g = StateGraph(State)
    g.add_node("auditor", auditor); g.add_node("refactorer", refactorer); g.add_node("qa", qa)
    g.add_edge(START, "auditor"); g.add_edge("auditor", "refactorer"); g.add_edge("refactorer", "qa")
    g.add_conditional_edges("qa", _route, {"done": END, "retry": "refactorer", "give_up": END})
    return g.compile()

In [ ]:
%%writefile debtbuster/src/debtbuster/cli.py
"""debtbuster: audit | fix — the whole UX in ~40 lines."""
import argparse, pathlib, sys
from .graph import build
from .tools import radon_avg_cc, pyexamine

def main() -> int:
    ap = argparse.ArgumentParser(prog="debtbuster",
                                 description="LLM-agent code auditing & gated refactoring")
    sub = ap.add_subparsers(dest="cmd", required=True)
    a = sub.add_parser("audit", help="static + agent audit of a file")
    a.add_argument("path")
    f = sub.add_parser("fix", help="run the auditor->refactorer<->QA team on a file")
    f.add_argument("path")
    f.add_argument("--tests", default=None, help="pytest file pinning behaviour (enables gate 2)")
    args = ap.parse_args()

    src = pathlib.Path(args.path).read_text()
    if args.cmd == "audit":
        print(f"avg cyclomatic complexity: {radon_avg_cc(args.path)}")
        print(pyexamine(str(pathlib.Path(args.path).parent))[:1500] or "(no PyExamine findings)")
        return 0

    team = build()
    out = team.invoke({"path": args.path, "test_file": args.tests, "source": src,
                       "findings": "", "candidate": "", "verdict": "",
                       "accepted": False, "iteration": 0})
    if out["accepted"]:
        dst = pathlib.Path(args.path).with_suffix(".refactored.py")
        dst.write_text(out["candidate"])
        print(f"ACCEPTED after {out['iteration']} iteration(s): {out['verdict']}\n-> {dst}")
        return 0
    print(f"REJECTED: {out['verdict'][:300]}\n(the gate held - nothing unverified shipped)")
    return 1

if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile debtbuster/src/debtbuster/__init__.py
"""debtbuster — lean LLM-agent harness from LLMA4SE Workshop 4."""
__version__ = "0.1.0"

In [ ]:
%%writefile debtbuster/tests/test_gates.py
"""The harness tests ITS OWN gates — the most load-bearing code gets the tests."""
from debtbuster.gates import verify

GOOD = "def f(x):\n    return x + 1\n"
BAD_SYNTAX = "def f(x:\n    return"

def test_gate1_rejects_broken_syntax(tmp_path):
    orig = tmp_path / "m.py"; orig.write_text(GOOD)
    v = verify(str(orig), BAD_SYNTAX, test_file=None)
    assert not v["ok"] and "gate 1" in v["why"]

def test_gates_accept_identical_code(tmp_path):
    orig = tmp_path / "m.py"; orig.write_text(GOOD)
    v = verify(str(orig), GOOD, test_file=None)
    assert v["ok"]

def test_gate3_rejects_complexity_regression(tmp_path):
    orig = tmp_path / "m.py"; orig.write_text(GOOD)
    worse = ("def f(x):\n"
             "    if x > 0:\n"
             "        if x > 1:\n"
             "            if x > 2:\n"
             "                return x\n"
             "    return x + 1\n")
    v = verify(str(orig), worse, test_file=None)
    assert not v["ok"] and "gate 3" in v["why"]

In [ ]:
# Install the harness in editable mode and prove it works — tests first, like adults
%pip install -q -e ./debtbuster
!cd debtbuster && python -m pytest tests/ -q

In [ ]:
# The CLI, on the same patient as Parts 1–3:
!debtbuster audit inventory.py
print("="*60)
!debtbuster fix inventory.py --tests test_inventory.py

📦 **That's the whole harness.** Five files, a config, its own tests, one CLI. `pip install`-able into any CI job (`debtbuster fix module.py --tests tests.py` → exit code 1 when the gate rejects, so the pipeline fails safely).

## 4.5 · ✍️ Exercise 4 (pick one)

**A. MLScent gate.** Add a `--ml` flag to `debtbuster fix` that adds a gate: *the MLScent smell count must not increase*. (You wrote this logic in Exercise 2D — now productionise it.)

**B. Deep-agent fixer.** Give `create_deep_agent` a `run_tests` tool and ask it to refactor `inventory.py` autonomously. Compare its success rate over 3 runs against the LangGraph team. Which would you put in CI, and why?

**C. Checkpointing.** Add `langgraph.checkpoint.memory.InMemorySaver` to `build()` and re-run with a `thread_id` — inspect the state history. You just rebuilt Part 2's flight recorder, for free.

**D. Model swap.** Change `config.OLLAMA_MODEL` to `qwen2.5-coder:3b`, re-run `debtbuster fix`. Measure: iterations to acceptance, wall time, rejection reasons. Write one sentence on the size/reliability trade-off.

In [ ]:
# 🖊️ Your Exercise 4 workspace


## 4.6 · Checkpoint ✅ & when to use what

1. Why does the QA gate stay identical across hand-rolled, LangGraph, and deep-agent versions? *(Because verification is a property of the **task**, not the framework.)*
2. What's the one-line change that would point this whole harness at a frontier cloud model? *(The model string in `config.py` / the `ChatOllama` constructor.)*
3. When would you choose a deep agent over a fixed graph? *(Exploratory, open-ended tasks; low blast radius. Fixed graphs + gates for anything repeatable that ships.)*

| Situation | Reach for |
|---|---|
| Understand the pattern / teach it | hand-rolled loop (Parts 1–3) |
| Repeatable pipeline, needs audit trail & CI | **LangGraph** graph + deterministic gates (`debtbuster`) |
| Open-ended exploration, human reviews output | **Deep agent** with tools |
| Serving many agents one local model | **Ollama** behind them all |

**🏁 End of the workshop.** You've built the same system three ways — which means you now *own* the pattern, not the framework. Ship the harness. Mind the gates. 🛡️